In [8]:
# ============================================================
# CELL 1 - SPEAKCHECK AI SETUP
# ============================================================

# Install Python packages only
!pip install -q SpeechRecognition pydub gradio numpy==2.2.6

print("✅ Packages installed successfully!")

✅ Packages installed successfully!


In [9]:
# ============================================================
# CELL 2 - VERIFY INSTALLATION
# ============================================================

import gradio as gr
import speech_recognition as sr
import numpy as np
import os
import tempfile
import re
import wave

from pydub import AudioSegment

print("======================================")
print("🎙️ SpeakCheck AI")
print("======================================")
print("NumPy:", np.__version__)
print("Gradio:", gr.__version__)
print("SpeechRecognition: ✅")
print("Pydub: ✅")
print("======================================")
print("🚀 Environment ready!")

🎙️ SpeakCheck AI
NumPy: 2.1.3
Gradio: 6.27.0
SpeechRecognition: ✅
Pydub: ✅
🚀 Environment ready!


In [12]:
# ============================================================
# CELL 3 - SPEAKCHECK AI
# Automatic Speech Recognition & Communication Analyzer
# ============================================================

import gradio as gr
import speech_recognition as sr
import numpy as np
import os
import tempfile
import re
import wave

from pydub import AudioSegment


# ============================================================
# AUDIO CONVERSION
# ============================================================

def convert_to_wav(audio_file):

    audio = AudioSegment.from_file(audio_file)

    # Standard format for ASR
    audio = audio.set_channels(1)
    audio = audio.set_frame_rate(16000)
    audio = audio.set_sample_width(2)

    wav_file = tempfile.NamedTemporaryFile(
        suffix=".wav",
        delete=False
    ).name

    audio.export(
        wav_file,
        format="wav"
    )

    return wav_file


# ============================================================
# AUDIO ANALYSIS
# ============================================================

def get_audio_info(wav_file):

    with wave.open(wav_file, "rb") as audio:

        frames = audio.getnframes()
        frame_rate = audio.getframerate()

        duration = frames / float(frame_rate)

        raw_data = audio.readframes(frames)

        samples = np.frombuffer(
            raw_data,
            dtype=np.int16
        )

        if len(samples) > 0:

            rms = np.sqrt(
                np.mean(
                    samples.astype(np.float64) ** 2
                )
            )

            voice_energy = rms / 32768.0

        else:

            voice_energy = 0

    return duration, voice_energy


# ============================================================
# AUTOMATIC SPEECH RECOGNITION
# ============================================================

def recognize_speech(wav_file):

    recognizer = sr.Recognizer()

    with sr.AudioFile(wav_file) as source:

        recognizer.adjust_for_ambient_noise(
            source,
            duration=0.5
        )

        audio_data = recognizer.record(
            source
        )

    try:

        text = recognizer.recognize_google(
            audio_data,
            language="en-US"
        )

        return text

    except sr.UnknownValueError:

        return "Speech could not be understood."

    except sr.RequestError:

        return "Speech recognition service is unavailable."


# ============================================================
# FILLER WORD DETECTION
# ============================================================

def detect_filler_words(text):

    filler_words = [
        "um",
        "uh",
        "hmm",
        "like",
        "actually",
        "basically",
        "you know",
        "okay"
    ]

    text_lower = text.lower()

    detected = []

    for word in filler_words:

        pattern = r"\b" + re.escape(word) + r"\b"

        count = len(
            re.findall(
                pattern,
                text_lower
            )
        )

        if count > 0:

            detected.append(
                f"{word}: {count}"
            )

    if not detected:

        return "No major filler words detected."

    return ", ".join(detected)


# ============================================================
# FLUENCY SCORE
# ============================================================

def calculate_fluency_score(
    speaking_rate,
    filler_result
):

    score = 100

    if speaking_rate < 80:

        score -= 20

    elif speaking_rate < 100:

        score -= 10

    elif speaking_rate > 190:

        score -= 20

    elif speaking_rate > 160:

        score -= 10

    if "No major filler words" not in filler_result:

        score -= 10

    return max(
        0,
        score
    )


# ============================================================
# MAIN SPEECH ANALYSIS
# ============================================================

def analyze_speech(audio):

    if audio is None:

        return (
            """
### ⚠️ No Audio

Please record or upload your speech.
""",
            "",
            ""
        )

    wav_file = None

    try:

        # Convert uploaded/recorded audio
        wav_file = convert_to_wav(audio)

        # Audio information
        duration, voice_energy = get_audio_info(
            wav_file
        )

        # Speech recognition
        transcript = recognize_speech(
            wav_file
        )

        # Count words
        words = re.findall(
            r"\b[a-zA-Z]+\b",
            transcript
        )

        word_count = len(words)

        # Speaking rate
        if duration > 0 and word_count > 0:

            speaking_rate = (
                word_count / duration
            ) * 60

        else:

            speaking_rate = 0

        # Speaking speed
        if speaking_rate < 100:

            speed = "Slow"

        elif speaking_rate <= 160:

            speed = "Normal"

        else:

            speed = "Fast"

        # Filler words
        filler_result = detect_filler_words(
            transcript
        )

        # Fluency score
        score = calculate_fluency_score(
            speaking_rate,
            filler_result
        )

        # Final result
        result = f"""
### 🎤 Speech Analysis

| Metric | Result |
|---|---|
| ⏱️ Duration | **{duration:.2f} seconds** |
| 📝 Detected Words | **{word_count}** |
| 🚀 Speaking Rate | **{speaking_rate:.2f} words/minute** |
| 🎯 Speaking Speed | **{speed}** |
| 🔊 Voice Energy | **{voice_energy:.4f}** |
| ⭐ Fluency Score | **{score}/100** |

---

### 🔤 Filler Word Analysis

**{filler_result}**
"""

        return (
            result,
            transcript,
            f"{score}/100"
        )

    except Exception as e:

        return (
            f"""
### ❌ Processing Error

**{str(e)}**

Please try recording again or upload
a clear audio file.
""",
            "",
            ""
        )

    finally:

        if (
            wav_file
            and os.path.exists(wav_file)
        ):

            os.remove(wav_file)


# ============================================================
# PROFESSIONAL CSS
# ============================================================

custom_css = """

body {
    background:
        linear-gradient(
            135deg,
            #eef2ff,
            #f5f3ff
        );
}

.gradio-container {
    max-width: 1150px !important;
    margin: auto;
}

.title {
    text-align: center;
    font-size: 42px;
    font-weight: 800;
    color: #4f46e5;
    margin-bottom: 5px;
}

.subtitle {
    text-align: center;
    font-size: 18px;
    color: #555;
    margin-bottom: 25px;
}

button {
    border-radius: 10px !important;
}

textarea,
input {
    border-radius: 12px !important;
}

"""


# ============================================================
# APPLICATION
# ============================================================

with gr.Blocks(
    css=custom_css,
    title="SpeakCheck AI"
) as app:

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    gr.Markdown(
        """
        <div class="title">
        🎙️ SpeakCheck AI
        </div>

        <div class="subtitle">
        Intelligent Automatic Speech Recognition
        & Communication Skill Analyzer
        </div>
        """
    )

    # --------------------------------------------------------
    # SPEECH ANALYZER
    # --------------------------------------------------------

    with gr.Tabs():

        with gr.Tab(
            "🎤 Speech Analyzer"
        ):

            gr.Markdown(
                """
                ### 🎙️ Record or Upload Your Speech

                Speak naturally and let SpeakCheck AI analyze
                your speech, speaking speed, filler words,
                voice energy and fluency.
                """
            )

            audio_input = gr.Audio(
                sources=[
                    "microphone",
                    "upload"
                ],
                type="filepath",
                label="🎙️ Record / Upload Speech"
            )

            analyze_button = gr.Button(
                "🔍 Analyze My Speech",
                variant="primary"
            )

            with gr.Row():

                analysis_output = gr.Markdown()

                score_output = gr.Textbox(
                    label="⭐ Fluency Score",
                    interactive=False
                )

            transcript_output = gr.Textbox(
                label="📝 Automatic Speech Recognition",
                lines=7,
                interactive=False
            )

            analyze_button.click(
                analyze_speech,
                inputs=audio_input,
                outputs=[
                    analysis_output,
                    transcript_output,
                    score_output
                ]
            )

    # --------------------------------------------------------
    # FOOTER
    # --------------------------------------------------------

    gr.Markdown(
        """
        ---

        ### 🚀 SpeakCheck AI

        **Automatic Speech Recognition • Speech Analysis •
        Speaking Rate • Filler Word Detection •
        Fluency Evaluation**

        An AI-powered communication analysis system.
        """
    )


# ============================================================
# LAUNCH
# ============================================================

app.launch(
    share=True,
    debug=True
)

/tmp/ipykernel_1114/1718248440.py:379: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://1b2b33806236637f1f.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://1b2b33806236637f1f.gradio.live
